# Tests with Port Angeles data

In [ ]:
import os
import sys

import dascore as dc
import h5py
import matplotlib.pyplot as plt
import numpy as np

# sys.path.append('../../')
sys.path.append(os.path.join(os.path.dirname(""), os.pardir, os.pardir))
import coherence_analysis.utils as f

## Using dascore

In [ ]:
# file= r"D:\CSM\Mines_Research\Test_data\Port_Angeles\Cascadia_DAS_1078DT_25PR_14GL_5DEC_2023-05-01T122349Z.h5"
# file= r"D:\CSM\Mines_Research\Test_data\Port_Angeles\Cascadia_DAS_1078DT_25PR_14GL_5DEC_2023-04-25T112349Z.h5"
file = r"D:\CSM\Mines_Research\Test_data\Port_Angeles\Cascadia_DAS_1078DT_25PR_14GL_5DEC_2023-06-01T074209Z.h5"

In [ ]:
patch = dc.spool(file)[0]
# patch = patch.detrend("distance")
patch = patch.pass_filter(time=(None, 200))

In [ ]:
patch.viz.waterfall(show=True, scale=0.1)

In [ ]:
data_path = r"D:\CSM\Mines_Research\Test_data\Port_Angeles"
spool = dc.spool(data_path)

In [ ]:
contents = spool.get_contents()
print(contents)

Set up parameters for coherence analysis

In [ ]:
averaging_window = 60
sub_window_length = 2
overlap = 0
samples_per_sec = 500
method = "exact"
start_channel = 1
n_channels = 200

Implement the coherence analysis

In [ ]:
# chunk the spool into averaging_window length
spool = spool.chunk(time=averaging_window)

# subselect n_channels number of channels starting from start_channel
sub_spool = spool.select(
    distance=(start_channel, start_channel + n_channels), samples=True
)

# another way to subselect channels
# sub_patch = patch.select(distance=np.array([0, 12, 10, 9]), samples=True)

# perform coherence calculation on each patch
map_out = sub_spool.map(
    lambda x: f.coherence(
        x.data.T,
        sub_window_length,
        overlap,
        sample_interval=1 / samples_per_sec,
        method=method,
    )
)

In [ ]:
sub_spool[0].viz.waterfall(show=True, scale=0.1)

In [ ]:
detection_significance = np.stack([a[0] for a in map_out], axis=-1)
eig_estimates = np.stack([a[1] for a in map_out], axis=-1)

In [ ]:
plt.imshow(detection_significance, aspect="auto", interpolation="none")
plt.colorbar()

In [ ]:
content = sub_spool.get_contents()
content[["time_min", "time_max"]]
content["time_step"][0].total_seconds()

In [ ]:
contents

## Exploring the h5 file

In [ ]:
f = h5py.File(file, "r")

In [ ]:
list(f.keys())

In [ ]:
list(f["Acquisition"].keys())

In [ ]:
f["Acquisition"]["Raw[0]"].keys()

In [ ]:
# f['Acquisition']['Raw[0]']['RawData']
f["Acquisition"]["Raw[0]"]["RawDataTime"]

In [ ]:
data = np.array(f["Acquisition"]["Raw[0]"]["RawData"])
timestamp_arr = np.array(f["Acquisition"]["Raw[0]"]["RawDataTime"])

In [ ]:
v_min = np.percentile(data, 0.5)
v_max = np.percentile(data, 99)
tick_size = 12
fsize = 14

fig2 = plt.figure()
img2 = plt.imshow(
    data,
    cmap="RdBu",
    vmin=v_min,
    vmax=v_max,
    aspect="auto",
    interpolation="none",
)
#   extent=(0, 90, 5000, 1))
plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Channels", fontsize=fsize)
# plt.title('Background noise',fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
cbar = plt.colorbar()
# cbar.ax.tick_params(labelsize=tick_size)

In [ ]:
timestamp_arr

In [ ]:
type(content["time_min"][0])

In [ ]:
from datetime import datetime

In [ ]:
# datetime.fromtimestamp(content['time_min'][0])
content["time_min"][content.index[-1]]  # .to_pydatetime()

In [ ]:
sub_spool.select(time=(content["time_min"][2].to_pydatetime(), ...))

In [ ]:
datetime_str = "06/01/23 07:32:09"
dt = datetime.strptime(datetime_str, "%m/%d/%y %H:%M:%S")

In [ ]:
str(dt).replace(" ", "_")

In [ ]:
sub_spool.select(time=[dt, ...])

In [ ]:
sub_spool.select(distance=(0, 1))[0].data.shape

In [ ]:
dl = ("06/01/23 07:32:09", ...)
dl = [
    datetime.strptime(a, "%m/%d/%y %H:%M:%S") if a != ... else ... for a in dl
]

In [ ]:
spool[0].data.shape[1]

In [ ]:
channel_range = (..., ...)
channels = np.arange(
    channel_range[0] if channel_range[0] is not ... else 0,
    channel_range[1]
    if channel_range[1] is not ...
    else spool[0].data.shape[1],
    1,
    dtype=int,
)

In [ ]:
channels